In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
sensitive_boundaries_results_path = "/scratch1/smaruj/sensitive_boundary_results.tsv"

In [3]:
df = pd.read_csv(sensitive_boundaries_results_path, sep="\t")

In [4]:
import ast

In [5]:
df["disrupted_bin"] = df["disrupted_bin"].apply(ast.literal_eval)

In [6]:
filtered_sorted_neg_df = (
    df[df["DeltaBoundaryStrength"] < 0]
    .sort_values(by="DeltaBoundaryStrength")
    .reset_index(drop=True)
)

In [ ]:
# filtered_sorted_pos_df = (
#     df[df["DeltaBoundaryStrength"] >= 0]
#     .sort_values(by="DeltaBoundaryStrength")
#     .reset_index(drop=True)
# )

In [7]:
to_check = filtered_sorted_neg_df[:10]

In [8]:
to_check

,chrom,start,end,window_end,window_start,disrupted_bin,SCD,DeltaBoundaryStrength
0,chr5,5470000,5480000,6131504,4820784,"[258, 264]",41.033791,-0.143692
1,chr13,53770000,53780000,54431504,53120784,"[255, 259, 250]",39.099865,-0.141888
2,chr18,76350000,76360000,77011504,75700784,"[252, 264, 265, 267]",36.822563,-0.123554
3,chr5,98340000,98350000,99001504,97690784,"[247, 255, 257, 264]",32.557678,-0.116049
4,chr4,118980000,118990000,119641504,118330784,"[253, 257, 258, 265]",45.330536,-0.113569
5,chr1,7230000,7240000,7891504,6580784,[247],34.867424,-0.110946
6,chr14,66350000,66360000,67011504,65700784,"[257, 266]",46.840149,-0.103644
7,chr3,44910000,44920000,45571504,44260784,"[254, 256, 266]",38.947369,-0.101955
8,chr13,74310000,74320000,74971504,73660784,"[256, 248, 241, 268]",25.958876,-0.091647
9,chr2,171410000,171420000,172071504,170760784,[256],29.601843,-0.089376


In [ ]:
from pyfaidx import Fasta

In [ ]:
fasta_file = "/project/fudenber_735/genomes/hg38/hg38.fa"

In [ ]:
genome = Fasta(fasta_file)

In [ ]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()
    
    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    valid_bases = list(base_to_int.keys())

    # Step 1: Convert sequence to integer encoding with random base for 'N'
    encoded_indices = []
    for base in sequence:
        if base in base_to_int:
            encoded_indices.append(base_to_int[base])
        else:
            random_base = random.choice(valid_bases)
            encoded_indices.append(base_to_int[random_base])

    # Step 2: One-hot encode the sequence
    encoded_sequence = np.array(encoded_indices)
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    # Step 3: Expand dimensions to [1, 4, sequence_length]
    # input_sequence = np.expand_dims(one_hot_encoded, axis=0)

    return one_hot_encoded

In [ ]:
from torch.utils.data import Dataset
import torch

In [ ]:
def permute_disrupted_bins(seq, row, bin_size=2048, cropping=64):
    # Make seq mutable
    seq = list(seq)
    
    for bin_idx in row["disrupted_bin"]:
        start = (bin_idx + cropping) * bin_size
        end = start + bin_size
        if end > len(seq):  # Avoid index error
            continue
        region = seq[start:end]
        np.random.shuffle(region)
        seq[start:end] = region  # Replace with shuffled region

    return ''.join(seq)

In [ ]:
class GenomicSequenceDataset(Dataset):
    def __init__(self, coord_df, genome_fasta, transform_fn=None):
        self.coords = coord_df  # DataFrame with chrom, start, end
        self.genome = genome_fasta
        self.transform_fn = transform_fn  # Optional function to modify sequence

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        TARGET_LEN = 1310720
        
        row = self.coords.iloc[idx]
        chrom, start, end = row["chrom"], row["window_start"], row["window_end"]
        seq = self.genome[chrom][start:end].seq.upper()
        
        # Fix sequence length if needed
        if len(seq) != TARGET_LEN:
            seq = seq[:TARGET_LEN].ljust(TARGET_LEN, 'N')  # pad with Ns if needed
        
        # Apply transformation, e.g. permute a window
        if self.transform_fn is not None:
            seq = self.transform_fn(seq, row)  # Pass row in case you want loc info
        
        one_hot = one_hot_encode_sequence(seq)  # shape: (4, L)
        return torch.from_numpy(one_hot.copy())

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
# Original dataset
orig_dataset = GenomicSequenceDataset(to_check, genome)

# Modified (permuted) dataset
perm_dataset = GenomicSequenceDataset(to_check, genome, transform_fn=permute_disrupted_bins)

orig_loader = DataLoader(orig_dataset, batch_size=10, shuffle=False)
perm_loader = DataLoader(perm_dataset, batch_size=10, shuffle=False)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

In [ ]:
import numpy as np
import torch

def set_diag(matrix, value, k):
    """Set diagonal `k` of a matrix to `value`."""
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu_batch(batch_vectors, matrix_len=512, num_diags=2):
    """Convert a batch of upper-triangular vectors into symmetric matrices with np.nan on diagonals."""
    if isinstance(batch_vectors, torch.Tensor):
        batch_vectors = batch_vectors.detach().cpu().numpy()

    batch_size = batch_vectors.shape[0]
    matrices = np.zeros((batch_size, matrix_len, matrix_len), dtype=np.float32)

    triu_indices = np.triu_indices(matrix_len, num_diags)

    for i in range(batch_size):
        matrices[i][triu_indices] = batch_vectors[i]
        # Mirror to lower triangle
        matrices[i] = matrices[i] + matrices[i].T

        # Set diagonals to np.nan
        for k in range(-num_diags + 1, num_diags):
            set_diag(matrices[i], np.nan, k)

    return matrices  # shape: [B, 512, 512]

In [ ]:
orig_preds_all = []
perm_preds_all = []

results = []

with torch.no_grad():
    for orig_batch, perm_batch in zip(orig_loader, perm_loader):
        orig_preds = model(orig_batch.to(device)).cpu()
        perm_preds = model(perm_batch.to(device)).cpu()
        
        scd = torch.sqrt(0.5 * ((perm_preds - orig_preds) ** 2).sum(dim=(1, 2)))
        print(scd)
        
        maps = from_upper_triu_batch(orig_preds - perm_preds)
        boundary_strength = np.nanmean(maps[:, 0:256, 256:512], axis=(1, 2))  # shape: [B]
        
        orig_preds_all.append(orig_preds)
        perm_preds_all.append(perm_preds)
        
        # Combine into results
        for i in range(len(scd)):
            results.append({
                "SCD": scd[i].item(),
                "DeltaBoundaryStrength": boundary_strength[i]
            })
        
        

In [ ]:
orig_preds_all = torch.cat(orig_preds_all, dim=0)
perm_preds_all = torch.cat(perm_preds_all, dim=0)

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
results_df

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
for i in range(10):
    print(i)
    matrix = from_upper_triu(orig_preds_all[i,:,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.axhline(256, color='black', linestyle='--', linewidth=1)
    plt.axvline(256, color='black', linestyle='--', linewidth=1)
    plt.colorbar()
    plt.show()
    

In [ ]:
for i in range(10):
    print(i)
    matrix = from_upper_triu(perm_preds_all[i,:,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.axhline(256, color='black', linestyle='--', linewidth=1)
    plt.axvline(256, color='black', linestyle='--', linewidth=1)
    plt.colorbar()
    plt.show()

In [ ]:
for i in range(10):
    print(i)
    matrix = from_upper_triu(orig_preds_all[i,:,:] - perm_preds_all[i,:,:], matrix_len=512, num_diags=2)
    
    plt.figure(figsize=(5, 5))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.axhline(256, color='black', linestyle='--', linewidth=1)
    plt.axvline(256, color='black', linestyle='--', linewidth=1)
    plt.colorbar()
    plt.show()